# BPMA probability heat map

Run the final cell to fit and plot every supported 2D dataset.

In [1]:
import sys
from pathlib import Path
import warnings

warnings.filterwarnings(
    "ignore",
    message="A worker stopped while some jobs were given to the executor.*",
    category=UserWarning,
)

project_root = Path.cwd()
if not (project_root / "bayesian_predictive_model_averaging").is_dir() and (project_root.parent / "bayesian_predictive_model_averaging").is_dir():
    project_root = project_root.parent
if (project_root / "bayesian_predictive_model_averaging").is_dir():
    sys.path.insert(0, str(project_root))

from scripts.classification_2d_demo import run_and_report
from IPython.display import display
import matplotlib.pyplot as plt

import numpy as np
from typing import Any

def plot_probability_heatmap(
    model: Any,
    X: np.ndarray,
    y: np.ndarray | None = None,
    *,
    X_train: np.ndarray | None = None,
    y_train: np.ndarray | None = None,
    padding: float = 0.55,
    grid_size: int = 250,
    title: str | None = None,
    ax: Any | None = None,
    colorbar: bool = True,
    decision_margin: float = 0.1,
):
    """Plot binary or multiclass probabilities over a 2D feature grid.

    Multiclass backgrounds use a probability-weighted ``tab10`` color mixture.
    Normalized entropy fades uncertain regions toward white, while black
    contours show predicted-class boundaries and the confidence boundary.
    The thin contour is controlled by ``decision_margin``. For binary plots it
    shows ``abs(p_1 - p_0) == decision_margin``; for multiclass plots it shows
    where the top-class probability exceeds the second-highest probability by
    ``decision_margin``. This makes the contour comparable across class counts.
    Matplotlib is imported lazily so plotting remains an optional dependency.
    """

    import matplotlib.pyplot as plt
    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import Normalize

    X = np.asarray(X)
    if X.ndim != 2 or X.shape[1] != 2:
        raise ValueError("X must have shape (n_samples, 2) for heatmap plotting")
    if grid_size < 2:
        raise ValueError("grid_size must be at least 2")
    if not np.isfinite(decision_margin) or not 0.0 <= decision_margin <= 1.0:
        raise ValueError("decision_margin must be finite and between 0 and 1")
    X_points = X if X_train is None else np.asarray(X_train)
    y_points = y if y_train is None else np.asarray(y_train)
    if y_points is None:
        raise ValueError("y or y_train is required to plot data points")

    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - padding, X[:, 0].max() + padding, grid_size),
        np.linspace(X[:, 1].min() - padding, X[:, 1].max() + padding, grid_size),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    probabilities = np.asarray(model.predict_proba(grid), dtype=float).reshape(
        xx.shape + (len(model.classes_),)
    )
    classes = np.asarray(model.classes_)
    cmap = plt.get_cmap("tab10")
    palette = np.asarray([cmap(index % 10)[:3] for index in range(len(classes))])
    predicted_indices = np.argmax(probabilities, axis=2)

    if len(classes) > 2:
        color_mixture = probabilities @ palette
        safe_probabilities = np.clip(probabilities, np.finfo(float).tiny, 1.0)
        entropy = -np.sum(probabilities * np.log(safe_probabilities), axis=2)
        confidence = 1.0 - entropy / np.log(len(classes))
        rgb = 1.0 - confidence[..., None] * (1.0 - color_mixture)
        if ax is None:
            fig, axis = plt.subplots(figsize=(9, 6.5))
        else:
            axis = ax
            fig = axis.figure
        axis.imshow(
            np.clip(rgb, 0.0, 1.0),
            origin="lower",
            extent=(xx.min(), xx.max(), yy.min(), yy.max()),
            interpolation="nearest",
            aspect="equal",
        )
        boundaries = np.arange(0.5, len(classes) - 0.5, 1.0)
        axis.contour(xx, yy, predicted_indices, levels=boundaries,
                     linewidths=2, colors="black")
        sorted_probabilities = np.sort(probabilities, axis=2)
        decision_margin_surface = sorted_probabilities[:, :, -1] - sorted_probabilities[:, :, -2]
        if decision_margin_surface.min() <= decision_margin <= decision_margin_surface.max():
            axis.contour(xx, yy, decision_margin_surface, levels=[decision_margin], linewidths=0.45,
                         colors="black")
        if colorbar:
            confidence_scale = ScalarMappable(norm=Normalize(0.0, 1.0), cmap="Greys_r")
            confidence_scale.set_array(confidence)
            confidence_bar = fig.colorbar(confidence_scale, ax=axis)
            confidence_bar.set_label("Confidence (1 − normalized entropy)")
    else:
        if ax is None:
            fig, axis = plt.subplots(figsize=(9, 6.5))
        else:
            axis = ax
            fig = axis.figure
        probability = probabilities[:, :, 1]
        axis.contourf(xx, yy, probability, levels=np.linspace(0, 1, 41),
                      cmap="RdBu_r", vmin=0, vmax=1, alpha=0.9)
        axis.contour(xx, yy, probability, levels=[0.5], linewidths=2.0,
                     colors="black")
        decision_margin_surface = np.abs(2.0 * probability - 1.0)
        if (
            decision_margin_surface.min()
            <= decision_margin
            <= decision_margin_surface.max()
        ):
            axis.contour(xx, yy, decision_margin_surface, levels=[decision_margin], linewidths=0.45,
                         colors="black")

    for class_index, class_value in enumerate(classes):
        mask = y_points == class_value
        axis.scatter(
            X_points[mask, 0], X_points[mask, 1], s=24,
            color=palette[class_index], edgecolors="white", linewidths=0.45,
            label=f"Class {class_value}",
        )
    axis.set_title(title or "Class probability heatmap")
    axis.set_xlabel("Feature 1")
    axis.set_ylabel("Feature 2")
    axis.set_aspect("equal", adjustable="box")
    axis.legend(loc="upper right")
    if ax is None:
        fig.tight_layout()
    return fig


In [2]:
DATASETS = (
    "circles",
    "moon",
    "spirals",
    "xor",
    "checkerboard",
    "gaussian",
    "blobs",
    "anisotropic_blobs",
    "classification",
)
BLOB_CENTER_RADIUS = 3.0
BLOB_CLUSTER_STANDARD_DEVIATION = 1.5

DATA_PARAMETERS = {
    "n_samples": 2000,
    "noise": 0.24,
    "random_state": 71,
    "feature_indices": (0, 1),
    "mean_distance": 3.0,
    "standard_deviation": 1.0,
    "n_classes": 4,
    "blob_center_radius": BLOB_CENTER_RADIUS,
    "blob_cluster_standard_deviation": BLOB_CLUSTER_STANDARD_DEVIATION,
    "circle_factor": 0.5,
    "spiral_turns": 2,
    "spiral_noise": 0.03,
    "checkerboard_cells": 4,
    "checkerboard_extent": 4.0,
    "checkerboard_label_noise": 0.0,
    "anisotropy": 3.0,
    "rotation": 0.35,
    "classification_class_sep": 1.0,
    "classification_flip_y": 0.05,
}

SPLIT_PARAMETERS = {
    "test_size": 0.30,
    "split_random_state": 7,
}

ESTIMATORS_PER_FAMILY = 100
ADAPTIVE_ROUND_SIZE_PER_FAMILY = 10

MODEL_PARAMETERS = {
    "temperature": 0.1,
    "n_estimators": "auto",
    "max_estimators": ESTIMATORS_PER_FAMILY,
    "tolerance": 0.001,
    "convergence_metric": "median",
    "convergence_size": 256,
    "cv": 5,
    "n_jobs": -1,
    "random_state": 12,
}

PLOT_PARAMETERS = {
    "padding": 0.55,
    "grid_size": 150,
    "decision_margin": 0.1,
}


In [ ]:
from bayesian_predictive_model_averaging import (
    RandomForestAdapter,
    FamilyRegistration,
    GaussianMixtureAdapter,
    KNNAdapter,
    LinearMixtureAdapter,
    MLPAdapter,
)

FAMILY_ADAPTERS = (
    ("random_forest", RandomForestAdapter),
#    ("linear_mixture", LinearMixtureAdapter),
#    ("gaussian_mixture", GaussianMixtureAdapter),
#    ("knn", KNNAdapter),
#    ("mlp", MLPAdapter),
)

for family_name, adapter_class in FAMILY_ADAPTERS:
    print(f"\n{'=' * 72}\n{family_name.upper()} ONLY\n{'=' * 72}")
    family_model_parameters = {
        **MODEL_PARAMETERS,
        "family_registry": [FamilyRegistration(adapter_class(), 1.0)],
    }
    for result in run_and_report(
        DATASETS,
        dataset_parameters=DATA_PARAMETERS,
        split_parameters=SPLIT_PARAMETERS,
        model_parameters=family_model_parameters,
    ):
        figure = plot_probability_heatmap(
            result.model,
            result.X,
            result.y,
            X_train=result.X_train,
            y_train=result.y_train,
            title=f"{family_name} — {result.dataset} — accuracy: {result.test_accuracy:.3f}",
            **PLOT_PARAMETERS,
        )
        display(figure)
        plt.close(figure)


RANDOM_FOREST ONLY


In [ ]:
ADAPTIVE_MODEL_PARAMETERS = {
    **MODEL_PARAMETERS,
    "max_estimators": ESTIMATORS_PER_FAMILY * len(FAMILY_ADAPTERS),
    "adaptive_importance_sampling": True,
    "round_size": ADAPTIVE_ROUND_SIZE_PER_FAMILY * len(FAMILY_ADAPTERS),
}

for result in run_and_report(
    DATASETS,
    dataset_parameters=DATA_PARAMETERS,
    split_parameters=SPLIT_PARAMETERS,
    model_parameters=ADAPTIVE_MODEL_PARAMETERS,
):
    figure = plot_probability_heatmap(
        result.model,
        result.X,
        result.y,
        X_train=result.X_train,
        y_train=result.y_train,
        title=f"All BPMA families — {result.dataset} — accuracy: {result.test_accuracy:.3f}",
        **PLOT_PARAMETERS,
    )
    display(figure)
    plt.close(figure)